# AI-07 Gradient boosting (XGBoost) and calibration

XGBoost when installed, otherwise HistGradientBoosting. Platt calibration is fitted on visitor-grouped training out-of-fold scores and kept only if validation Brier improves. The served model, its threshold and its confusion matrix are below.

Outputs come from `python -m ai.training.run --optional-models`; see `reports/full-report/ml_results.md` for the interpretation.

In [ ]:
# Setup: run from anywhere inside the repo
import json, pathlib, sys
import pandas as pd
ROOT = pathlib.Path.cwd()
while not (ROOT / "ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
EVAL = ROOT / "ai" / "evaluation"
pd.set_option("display.width", 160)

# Set to True to retrain everything on data/processed/*.parquet (5-15 min).
# False = read the committed outputs of the last real run.
RERUN = False
if RERUN:
    from ai.training.run import run
    run(include_optional=True)

summary = json.loads((EVAL / "run_summary.json").read_text(encoding="utf-8"))
comparison = pd.read_csv(EVAL / "model_comparison.csv")
tuning = pd.read_csv(EVAL / "tuning.csv")
print("model:", summary["selected_model"]["model_version"], "| split:", summary["split_version"])
print("scenario:", summary["scenario_id"], "| label task:", summary["label_task"])

In [ ]:
tuning[tuning.model.isin(["xgboost", "hist_gradient_boosting"])].sort_values("val_pr_auc", ascending=False)

In [ ]:
print(json.dumps(summary["selected_model"], indent=2))
print(json.dumps(summary["threshold"], indent=2))

In [ ]:
t = summary["test"]
print({k: t[k] for k in ["support", "positives", "prevalence", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier"]})
print("confusion [[TN, FP], [FN, TP]] =", t["confusion_matrix"])

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(EVAL / "figures" / "model_pr_roc.png")))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(EVAL / "figures" / "model_calibration.png")))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(EVAL / "figures" / "model_confusion_matrix.png")))

In [ ]:
pd.read_csv(EVAL / "threshold_study.csv").iloc[::5]

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(EVAL / "figures" / "model_threshold.png")))